# Testing LinBin flow for DEIMOS data

Set imports and paths

In [ ]:
from eregion.tasks import ImageCreator, AssembleFocalPlane
from eregion.tasks.custom import guess_image_type_from_filename_DEIMOS, load_image_fits_DEIMOS
from eregion.tasks.calibration import MasterBias, CalibrationResult, MasterCombine
from eregion.tasks.preprocessing import BiasSubtraction, ScanSubtraction, SigmaClipMasking
from eregion.tasks.linbin import LinBin, LinBinResult

import numpy as np
import os
import glob2
import importlib
import matplotlib.pyplot as plt
from functools import partial

In [ ]:
basepath = '/Users/yashvi/Desktop/Detector Characterization Tools/DTU_dettest/DTU_fullfp_bringup/bintest/SCI/'
inp = os.path.join(basepath, '20260722-171930')
out = inp.replace('DTU_dettest', 'DTU_detreduce')
os.makedirs(out, exist_ok=True)

Load images. Looking at image type table for the loaded data, images relevant for this notebook are bias, linbin, and flat, so we filter and keep only those.

In [ ]:
imtask = ImageCreator(detector_config='../src/eregion/configs/detectors/deimos_sci.yaml')
imres = imtask.run(input_source=inp, identifier_func=guess_image_type_from_filename_DEIMOS, fileloader_func=load_image_fits_DEIMOS, data_on_demand=True)
bundle = imres.data('type == "bias" | type =="linbin" | type == "flat"')
del imres
bundle

Run overscan subtraction on all.

In [ ]:
osubtask = ScanSubtraction(which_scan='serial_overscan', method='median_by_axis', trim_start=6)
imres = osubtask.run(images=bundle)

Make master biases

In [ ]:
biasbundle = imres.data('type == "bias" | (type =="linbin" & extra_0 == "bias")')
mbtask = MasterBias(groupby_keys=["det_id", "type", "extra_0"])
calres = mbtask.run(images=biasbundle)

Run sigma clipping on non-bias frames. For normal flats the sigma-clip is on the whole output, for linbin flats the sigms-clip is done per parallel axis index.

In [ ]:
flats = imres.data('type == "flat"')
crtask = SigmaClipMasking(sigma_clip_args={'sigma':5.0})
flatres = crtask.run(images=flats)

linbins = imres.data('type == "linbin" & extra_0 != "bias"')
crtask = SigmaClipMasking(sigma_clip_args={'sigma':5.0}, clip_axis="serial")
linbinres = crtask.run(images=linbins)

In [ ]:
# linbinres.data[0].show()

Run bias subtraction on non-bias frames.

In [ ]:
bsubtask = BiasSubtraction(only_image_area=True)
flatres = bsubtask.run(images=flats, master_bias=calres.master_bias('type == "master_bias_nan"'))
linbinres = bsubtask.run(images=linbins, master_bias=calres.master_bias('type == "master_linbin_bias"'))

Combine multiple frames of same type. There are 2 flats and 2 linbins per detector.

In [ ]:
combtask = MasterCombine(groupby_keys=['det_id', 'type'])
flatres = combtask.run(images=flatres.data)
linbinres = combtask.run(images=linbinres.data)

In [ ]:
from eregion.tasks import linbin
import importlib
importlib.reload(linbin)

lbtask = linbin.LinBin(groupby_keys=['det_id'])
lbres = lbtask.run(normal_flats=flatres.data, linbin_flats=linbinres.data)
df = lbres.stats
df

In [ ]:
def _plot_panels(ax, df, y_cols, x_cols, yscale, xscale):
    from matplotlib.lines import Line2D

    axkey = {"det_1": ax[0,3], "det_2": ax[0,2], "det_3": ax[0,1], "det_4": ax[0,0],
             "det_5": ax[1,0], "det_6": ax[1,1], "det_7": ax[1,2], "det_8": ax[1,3]}
    # get matplotlib's marker list
    markers_dict = Line2D.markers
    markers_list = [m for m in markers_dict.keys() if isinstance(m, str) and m not in [' ', 'None', '', 'none']]

    for det_id, axs in axkey.items():
        axs.clear()
        for out, ls in zip(['E', 'F'], ['--','dotted']):
            dfsub = df[(df['det_id']==det_id) & (df['output']==out)].iloc[0]
            for i,col in enumerate(y_cols):
                axs.plot(list(dfsub[x_cols[i]])[:-1], list(dfsub[col])[:-1], ls=ls, label=f'{col}-{x_cols[i]}, Chan-{out}',
                         marker=markers_list[i], alpha=0.7)

        axs.set_xlabel(','.join(np.unique(x_cols)), fontsize=10)
        axs.set_ylabel(','.join(np.unique(y_cols)), fontsize=10)
        axs.set_title(det_id, fontsize=12)
        axs.grid(True)
        try:
            axs.set_xscale(xscale)
        except:
            pass
        try:
            axs.set_yscale(yscale)
        except:
            pass
    ax[0, 3].legend(ncols=1, loc='upper left', bbox_to_anchor=(0.02, 1.1), fontsize=10, borderaxespad=0)
    return ax

fig, ax = plt.subplots(2, 4, figsize=(20,8), tight_layout=True)
ax = _plot_panels(ax, df, ["mean_analog_0", "mean_digital_0"], x_cols=["bins", "bins"], yscale='linear', xscale='linear')

In [ ]:
fig, ax = plt.subplots(2, 4, figsize=(20,8), tight_layout=True)
ax = _plot_panels(ax, df, ["mean_digital_0"], x_cols=["mean_analog_0"], yscale='linear', xscale='linear')

### Testing LinBin YAML flow

In [ ]:
from eregion.pipeline.engine import PipelineEngine

dirpath = '../../DTU_dettest/DTU_fullfp_bringup/bintest/SCI/20260722-171930'
eng = PipelineEngine(pipeline_config_input='../src/eregion/configs/pipeline_flows/linbin_flow.yaml',
                     runtime_variables={
                         'DETECTOR_CONFIG': '../src/eregion/configs/detectors/deimos_sci.yaml',
                         'BIAS_INPUT_SOURCE': dirpath+'/*bias*',
                         'LINBIN_INPUT_SOURCE': dirpath+'/*linbin*',
                         'FLAT_INPUT_SOURCE': dirpath+'/*flat*',
                     })

In [ ]:
eng.run()

In [ ]:
df = eng.results['digital_flow.linbin_stats'].stats
fig, ax = plt.subplots(2, 4, figsize=(20,8), tight_layout=True)
ax = _plot_panels(ax, df, ["mean_analog_0", "mean_digital_0"], x_cols=["bins", "bins"], yscale='linear', xscale='linear')